# Session 5: Practical Assessment – Advanced Anomaly Detection

**Course:** Machine Learning III (Unsupervised Learning) @Albert School  
**Format:** Groups of 1 to 3 students.  
**Duration:** 3 hours (Due at the end of the session).  
**Grading:** Graded (Low-impact, incentive-based).

### 📖 The Business Scenario
You are the Lead Data Science team for a major manufacturing firm. The company operates expensive, heavy machinery that occasionally suffers from catastrophic failures, halting production and costing **€100,000 per hour** of downtime. 

Your operations team has provided you with telemetry data from these machines (temperatures, torque, tool wear, etc.). Standard rules-based monitoring is no longer sufficient. Your objective is to build an unsupervised anomaly detection pipeline to flag potential machine failures *before* they occur, while minimizing "Alert Fatigue" (False Positives) for the maintenance crew.

### 🎯 Instructions & Deliverables
You must complete this notebook by addressing two distinct perspectives: the **Technical Data Scientist** and the **Business Manager**.

1. **Part 1: Exploratory Data Analysis (EDA) & Cleaning**
   - Investigate features, missing values, and distributions.
   - Preprocess the data (Standardization, handling categorical variables like `Type`).
2. **Part 2: Modeling & Hyperparameter Tuning**
   - Train 4 models: `IsolationForest`, `OneClassSVM`, `LocalOutlierFactor`, and `EllipticEnvelope`.
   - **Rule:** You must tune the trade-off parameters (`contamination`, `nu`, etc.) and justify your choices.
3. **Part 3: Technical Comparison & Visualizations**
   - Use PCA or t-SNE to project the data into 2D/3D.
   - Overlay the anomalies flagged by your models. 
   - Deep Dive: Isolate specific machines flagged by LOF but missed by iForest (or vice versa) and explain *why* based on the algorithm's mathematical assumptions.
4. **Part 4: Managerial Conclusion & Actionable Strategy**
   - **Cost Matrix:** A False Positive costs **€500**. A False Negative costs **€15,000**.
   - Bring back the `Machine failure` labels (hidden during training) and evaluate your models.
   - Conclude: Which model saves the company the most money?


In [1]:
# ==========================================
# 🚀 INITIALIZATION & DATA LOADING
# Run this cell to get started!
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# 1. Load the AI4I 2020 Predictive Maintenance Dataset directly from UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
print("Downloading dataset...")
df_raw = pd.read_csv(url)

print(f"Dataset loaded successfully! Shape: {df_raw.shape}")
display(df_raw.head())


Dataset loaded successfully! Shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


---
## Part 1: Exploratory Data Analysis (EDA) & Cleaning
*(Your code and analysis here. Think about standardization and how to handle the `Type` column!)*


In [ ]:
# ==========================================
# PART 1: EDA & CLEANING
# ==========================================

from sklearn.preprocessing import OrdinalEncoder, StandardScaler

# --- 1.1 Basic exploration ---
print("=== Shape ===")
print(df_raw.shape)

print("\n=== Data types ===")
print(df_raw.dtypes)

print("\n=== Missing values ===")
print(df_raw.isnull().sum())

print("\n=== Descriptive statistics ===")
display(df_raw.describe())

# --- 1.2 Distributions & skewness ---
num_features = [
    "Air temperature [K]", "Process temperature [K]",
    "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(num_features):
    axes[i].hist(df_raw[col], bins=40, edgecolor="black", color="steelblue")
    axes[i].set_title(f"{col}\nskewness: {df_raw[col].skew():.2f}")
    axes[i].set_xlabel(col)
axes[-1].axis("off")
plt.suptitle("Feature Distributions", fontsize=14)
plt.tight_layout()
plt.show()

print("\n=== Skewness ===")
print(df_raw[num_features].skew())

# --- Correlation heatmap ---
plt.figure(figsize=(8, 6))
sns.heatmap(df_raw[num_features].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

# --- 1.3 Separate y_true (used only in Part 4) ---
y_true = df_raw["Machine failure"].copy()
failure_rate = y_true.mean()
print(f"\n=== Failure rate: {failure_rate:.4f} ({y_true.sum()} failures / {len(y_true)} obs) ===")
print("→ Contamination parameter set to ~3.4%")

# Drop Machine failure + sub-failure labels + non-predictive identifiers
cols_to_drop = ["Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF", "UDI", "Product ID"]
df_train = df_raw.drop(columns=cols_to_drop).copy()
print(f"\nTraining features: {list(df_train.columns)}")

# --- 1.4 Encode categorical 'Type' (L / M / H) ---
# OrdinalEncoder preserves the quality ordering (L < M < H).
# Mandatory for distance-based models (LOF, Elliptic Envelope) which require numeric input.
enc = OrdinalEncoder(categories=[["L", "M", "H"]])
df_train["Type"] = enc.fit_transform(df_train[["Type"]])
print("\n=== Type encoding (0=L, 1=M, 2=H) ===")
print(df_train["Type"].value_counts().sort_index())

# --- 1.5 Standardisation ---
# Mandatory for LOF and Elliptic Envelope: RPM (~1000-2500) would dominate
# Torque (~3-80) without scaling, biasing all distance calculations.
# Not strictly required for Isolation Forest (random axis splits), but applied for consistency.
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(df_train), columns=df_train.columns)

print("\n=== Scaled features — mean ~0, std ~1 ===")
display(X_scaled.describe().loc[["mean", "std"]].round(2))
print("\n✅ Part 1 complete — X_scaled and y_true are ready")


---
## Part 2: Modeling & Hyperparameter Tuning
*(Train IsolationForest, OneClassSVM, LocalOutlierFactor, and EllipticEnvelope. Remember to tune your threshold parameters!)*


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope

# ==========================================
# PART 2: MODELING & HYPERPARAMETER TUNING
# ==========================================

# Contamination = failure rate derived from EDA (339 / 10 000 = 3.4%)
CONTAMINATION = round(float(y_true.mean()), 4)
print(f"Contamination rate (from EDA): {CONTAMINATION} ({CONTAMINATION*100:.1f}%)")

# --- 2.1 Isolation Forest ---
# n_estimators=200: more robust than default 100 on a 10k dataset
# contamination=0.034: anchored to the observed failure rate
iso = IsolationForest(n_estimators=200, contamination=CONTAMINATION, random_state=42)
pred_iso = iso.fit_predict(X_scaled)

# --- 2.2 One-Class SVM ---
# nu=0.034: upper bound on anomaly fraction (analogous to contamination)
# kernel="rbf": captures non-linear separations in sensor space
# gamma="scale": auto-adapts to feature variance (1 / (n_features × X.var()))
ocsvm = OneClassSVM(nu=CONTAMINATION, kernel="rbf", gamma="scale")
pred_svm = ocsvm.fit_predict(X_scaled)

# --- 2.3 Local Outlier Factor ---
# n_neighbors=20: balances noise sensitivity vs locality on this dense dataset
# LOF measures local density deviation → effective for isolated failure clusters
lof = LocalOutlierFactor(n_neighbors=20, contamination=CONTAMINATION)
pred_lof = lof.fit_predict(X_scaled)

# --- 2.4 Elliptic Envelope (Robust Covariance) ---
# support_fraction=0.85: fits covariance on 85% of points, robust to outliers
# Assumes multivariate Gaussian distribution — a known limitation worth noting in Part 3
ee = EllipticEnvelope(contamination=CONTAMINATION, support_fraction=0.85, random_state=42)
pred_ee = ee.fit_predict(X_scaled)

# --- Summary ---
results = pd.DataFrame({
    "IsolationForest": pred_iso,
    "OneClassSVM":     pred_svm,
    "LOF":             pred_lof,
    "EllipticEnvelope": pred_ee,
    "y_true":          y_true.values
})

anomaly_counts = (results.drop(columns="y_true") == -1).sum()
print("\n=== Anomalies flagged per model (-1 = anomaly) ===")
print(anomaly_counts.to_string())
print(f"\nExpected ~{int(CONTAMINATION * len(results))} anomalies ({CONTAMINATION*100:.1f}% × {len(results)})")
print("\n✅ Part 2 complete — predictions stored in 'results'")


---
## Part 3: Technical Comparison & Visualizations
*(Use PCA/t-SNE to visualize the flagged anomalies. Find an anomaly caught by one model but missed by another and explain why.)*


In [ ]:
from sklearn.decomposition import PCA

# Your code here


---
## Part 4: Managerial Conclusion & Business Strategy
*(Bring back `y_true`. Calculate the number of False Positives and False Negatives for each model. Apply the cost matrix. Which model wins?)*


In [ ]:
# Business Cost Matrix
COST_FP = 500     # False Positive: Wasted technician check
COST_FN = 15000   # False Negative: Catastrophic machine breakdown

# Your code here
